In [ ]:
# === Setup ===
# Runtime: <1m with OAI_FAST_MODE=1
# Hardware: CPU smoke
# Network: none
# Competition-safe: Yes for the declared profile
import os, random, math, re, json, csv, time
from pathlib import Path
import numpy as np

FAST_MODE = os.getenv("OAI_FAST_MODE", "0") == "1"
RUNTIME_PROFILE = os.getenv("OAI_RUNTIME_PROFILE", "cpu")
random.seed(42)
np.random.seed(42)
print(f"Runtime profile: {'fast' if FAST_MODE else 'full'}")

# Logistic Regression bằng NumPy

In [ ]:
rng = np.random.default_rng(42)
n = 100 if FAST_MODE else 400

# Generate synthetic data
X = np.r_[rng.normal([-1, -1], 0.7, (n // 2, 2)), rng.normal([1, 1], 0.7, (n // 2, 2))]
y = np.r_[np.zeros(n // 2), np.ones(n // 2)]


def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -30, 30)))


# Initialize parameters
w = np.zeros(2)
b = 0.0
lr = 0.2

# Training loop
for _ in range(150 if FAST_MODE else 800):
    p = sigmoid(X @ w + b)
    error = p - y
    w -= lr * (X.T @ error / len(X))
    b -= lr * error.mean()

# Evaluate
p = sigmoid(X @ w + b)
loss = -np.mean(y * np.log(p + 1e-9) + (1 - y) * np.log(1 - p + 1e-9))
acc = ((p >= 0.5) == y).mean()

assert loss < 0.3 and acc > 0.9
print("loss/accuracy", loss, acc, "w/b", w, b)

In [ ]:
# Gradient check w[0]
p = sigmoid(X @ w + b)
analytic = (X.T @ (p - y) / len(X))[0]
eps = 1e-5


def objective(w0):
    ww = w.copy()
    ww[0] = w0
    q = sigmoid(X @ ww + b)
    return -np.mean(y * np.log(q + 1e-9) + (1 - y) * np.log(1 - q + 1e-9))


numeric = (objective(w[0] + eps) - objective(w[0] - eps)) / (2 * eps)
assert abs(analytic - numeric) < 1e-5
print("gradient check", analytic, numeric)